In [0]:
%run ./03-invoice-stream

In [0]:
class invoiceStreamTestSuite():
    def __init__(self):
        import os
        self.base_data_dir = os.path.abspath("..")
        self.checkpoint_dir = ("/Volumes/spark_stream/default/streaming_data/checkpoints")

    def cleanTests(self):
        print(f"Starting Cleanup...", end='')
        spark.sql("drop table if exists spark_stream.default.invoice_line_items")

        dbutils.fs.rm(f"{self.checkpoint_dir}/invoice_line_items",True)
        dbutils.fs.rm(f"{self.base_data_dir}/data/invoices", True)

        dbutils.fs.mkdirs(f"{self.base_data_dir}/data/invoices")
        print("Done")

    def ingestData(self, itr):
        print(f"\tStarting Ingestion...", end='')
        dbutils.fs.cp(f"{self.base_data_dir}/datasets/invoices/invoices-{itr}.json", f"{self.base_data_dir}/data/invoices/")
        print("Done")

    def assertResult(self, expected_count):
        print(f"\tStarting validation...", end='')
        actual_count = spark.sql('''select count(*) 
                                 from spark_stream.default.invoice_line_items
                                 ''').collect()[0][0]
        assert expected_count == actual_count, f"Test failed! actual count is {actual_count}"
        print("Done")

        
    def runTests(self):
        self.cleanTests()

        iStream = invoiceStream()

        print("Testing first iteration of invoice stream...")
        self.ingestData(1)

        streamQuery = iStream.process()
        streamQuery.awaitTermination()

        self.assertResult(1253)
        print("Validation passed.\n")

        print("Testing second iteration of invoice stream...")
        self.ingestData(2)

        streamQuery = iStream.process()
        streamQuery.awaitTermination()

        self.assertResult(2510)
        print("Validation passed.\n")

        print("Testing third iteration of invoice stream...")
        self.ingestData(3)

        streamQuery = iStream.process()
        streamQuery.awaitTermination()

        self.assertResult(3994)
        print("Validation passed.\n")

        print("Testing fourth iteration of invoice stream...")
        self.ingestData(4)

        streamQuery = iStream.process()
        streamQuery.awaitTermination()

        self.assertResult(4016)
        print("Validation passed.\n")


In [0]:
isTS = invoiceStreamTestSuite()
isTS.runTests()	